In [2]:
# ============================================================
# IMPORT REQUIRED LIBRARIES
# ============================================================

import pandas as pd
import numpy as np

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import (
    r2_score,
    mean_squared_error
)

import joblib


print("All libraries imported successfully.")

All libraries imported successfully.


### Load the regression dataset

In [3]:
# ============================================================
# LOAD REGRESSION DATASET
# ============================================================

data_path = "../data/processed/model_data_regression.csv"

df = pd.read_csv(data_path)


print("Regression dataset loaded successfully.")
print("Dataset shape:", df.shape)

display(df.head())

Regression dataset loaded successfully.
Dataset shape: (52930, 87)


,VisitDate,VisitYear,VisitMonth,UserVisitCount,AttractionVisitCount,Rating,Country_Frequency,CityName_Frequency,VisitMode_Business,VisitMode_Couples,...,Attraction_Sempu Island,Attraction_Sewu Temple,Attraction_Tanah Lot Temple,Attraction_Tegalalang Rice Terrace,Attraction_Tegenungan Waterfall,Attraction_Ullen Sentalu Museum,Attraction_Uluwatu Temple,Attraction_Water Castle (Tamansari),Attraction_Waterbom Bali,Attraction_Yogyakarta Palace
0,2022-10-01,2022,10,1,13198,5,0.126998,0.000756,0,1,...,0,0,0,0,0,0,0,0,0,0
1,2022-10-01,2022,10,1,13198,5,0.028075,0.000208,0,0,...,0,0,0,0,0,0,0,0,0,0
2,2022-10-01,2022,10,1,13198,5,0.002645,0.000057,0,0,...,0,0,0,0,0,0,0,0,0,0
3,2022-10-01,2022,10,2,13198,3,0.004893,0.001512,0,0,...,0,0,0,0,0,0,0,0,0,0
4,2022-10-01,2022,10,3,13198,3,0.126998,0.003855,0,1,...,0,0,0,0,0,0,0,0,0,0


### Inspect the dataset

In [4]:
# ============================================================
# BASIC DATASET INSPECTION
# ============================================================

print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes.value_counts())

Number of rows: 52930
Number of columns: 87

Column names:
['VisitDate', 'VisitYear', 'VisitMonth', 'UserVisitCount', 'AttractionVisitCount', 'Rating', 'Country_Frequency', 'CityName_Frequency', 'VisitMode_Business', 'VisitMode_Couples', 'VisitMode_Family', 'VisitMode_Friends', 'VisitMode_Solo', 'Continent_Africa', 'Continent_America', 'Continent_Asia', 'Continent_Australia & Oceania', 'Continent_Europe', 'Region_Australia', 'Region_Caribbean', 'Region_Central Africa', 'Region_Central America', 'Region_Central Asia', 'Region_Central Europe', 'Region_East Africa', 'Region_East Asia', 'Region_Eastern Europe', 'Region_Middle East', 'Region_North Africa', 'Region_Northern America', 'Region_Northern Europe', 'Region_Oceania', 'Region_South America', 'Region_South Asia', 'Region_South East Asia', 'Region_Southern Africa', 'Region_Southern Europe', 'Region_Unknown', 'Region_West Africa', 'Region_Western Europe', 'AttractionType_Ancient Ruins', 'AttractionType_Ballets', 'AttractionType_Beaches

### Separate features and target

In [5]:
# ============================================================
# SEPARATE FEATURES (X) AND TARGET (y)
# ============================================================

# Target variable
y = df["Rating"]


# Input features
X = df.drop(
    columns=["Rating"]
)


print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)

print("\nTarget statistics:")
display(y.describe())

Feature matrix shape: (52930, 86)
Target shape: (52930,)

Target statistics:


count    52930.000000
mean         4.157699
std          0.970543
min          1.000000
25%          4.000000
50%          4.000000
75%          5.000000
max          5.000000
Name: Rating, dtype: float64

### Check the target

In [6]:
# ============================================================
# CHECK TARGET VARIABLE
# ============================================================

print("Minimum rating:", y.min())
print("Maximum rating:", y.max())

print("\nRating distribution:")
print(y.value_counts().sort_index())

Minimum rating: 1
Maximum rating: 5

Rating distribution:
Rating
1     1263
2     2035
3     7730
4    17966
5    23936
Name: count, dtype: int64


### Make sure all features are numeric

In [7]:
# ============================================================
# PREPARE FEATURES FOR MACHINE LEARNING
# ============================================================

# Convert every feature column to numeric.
# If something cannot be converted, it becomes NaN.
X = X.apply(
    pd.to_numeric,
    errors="coerce"
)


# Replace missing values with 0.
X = X.fillna(0)


# Replace infinite values with 0.
X = X.replace(
    [np.inf, -np.inf],
    0
)


print("Feature preparation completed.")

print(
    "Missing values:",
    X.isnull().sum().sum()
)

print(
    "Infinite values:",
    np.isinf(X.to_numpy()).sum()
)

Feature preparation completed.
Missing values: 0
Infinite values: 0


### Train/Test split

In [8]:
# ============================================================
# TRAIN / TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)


print("Training data:")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("\nTesting data:")
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

Training data:
X_train: (42344, 86)
y_train: (42344,)

Testing data:
X_test: (10586, 86)
y_test: (10586,)


### Create a function to evaluate models

In [9]:
# ============================================================
# MODEL EVALUATION FUNCTION
# ============================================================

def evaluate_regression_model(model, X_test, y_test):
    """
    Evaluate a regression model using:

    R²   -> Higher is better
    MSE  -> Lower is better
    RMSE -> Lower is better
    """

    # Generate predictions
    predictions = model.predict(X_test)

    # Calculate R²
    r2 = r2_score(
        y_test,
        predictions
    )

    # Calculate Mean Squared Error
    mse = mean_squared_error(
        y_test,
        predictions
    )

    # Calculate Root Mean Squared Error
    rmse = np.sqrt(mse)

    return r2, mse, rmse

### Train Linear Regression

In [10]:
# ============================================================
# LINEAR REGRESSION BASELINE
# ============================================================

linear_model = Pipeline(
    steps=[
        (
            "scaler",
            StandardScaler()
        ),
        (
            "model",
            LinearRegression()
        )
    ]
)


print("Training Linear Regression...")

linear_model.fit(
    X_train,
    y_train
)


print("Linear Regression training completed.")

Training Linear Regression...
Linear Regression training completed.


In [11]:
# ============================================================
# EVALUATE LINEAR REGRESSION
# ============================================================

linear_r2, linear_mse, linear_rmse = (
    evaluate_regression_model(
        linear_model,
        X_test,
        y_test
    )
)


print("Linear Regression Results")
print("-------------------------")
print(f"R²   : {linear_r2:.4f}")
print(f"MSE  : {linear_mse:.4f}")
print(f"RMSE : {linear_rmse:.4f}")

Linear Regression Results
-------------------------
R²   : 0.0960
MSE  : 0.8514
RMSE : 0.9227


### Train Random Forest

In [12]:
# ============================================================
# RANDOM FOREST REGRESSOR
# ============================================================

random_forest = RandomForestRegressor(
    n_estimators=100,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)


print("Training Random Forest...")

random_forest.fit(
    X_train,
    y_train
)


print("Random Forest training completed.")

Training Random Forest...
Random Forest training completed.


In [13]:
# ============================================================
# DAY 4 - STEP 13
# EVALUATE RANDOM FOREST
# ============================================================

rf_r2, rf_mse, rf_rmse = (
    evaluate_regression_model(
        random_forest,
        X_test,
        y_test
    )
)


print("Random Forest Results")
print("---------------------")
print(f"R²   : {rf_r2:.4f}")
print(f"MSE  : {rf_mse:.4f}")
print(f"RMSE : {rf_rmse:.4f}")

Random Forest Results
---------------------
R²   : 0.1412
MSE  : 0.8089
RMSE : 0.8994


In [14]:
# ============================================================
# CHECK XGBOOST INSTALLATION
# ============================================================

try:
    import xgboost as xgb

    print(
        "XGBoost is installed."
    )

    print(
        "Version:",
        xgb.__version__
    )

    xgboost_available = True

except ImportError:

    print(
        "XGBoost is not installed."
    )

    xgboost_available = False

XGBoost is installed.
Version: 3.4.1


### Train XGBoost

In [15]:
# ============================================================
# XGBOOST REGRESSOR
# ============================================================

if xgboost_available:

    xgboost_model = xgb.XGBRegressor(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    )


    print("Training XGBoost...")

    xgboost_model.fit(
        X_train,
        y_train
    )


    print(
        "XGBoost training completed."
    )

else:

    print(
        "XGBoost training skipped."
    )

Training XGBoost...
XGBoost training completed.


In [16]:
# ============================================================
# EVALUATE XGBOOST
# ============================================================

if xgboost_available:

    xgb_r2, xgb_mse, xgb_rmse = (
        evaluate_regression_model(
            xgboost_model,
            X_test,
            y_test
        )
    )


    print("XGBoost Results")
    print("----------------")
    print(f"R²   : {xgb_r2:.4f}")
    print(f"MSE  : {xgb_mse:.4f}")
    print(f"RMSE : {xgb_rmse:.4f}")

else:

    print(
        "XGBoost evaluation skipped."
    )

XGBoost Results
----------------
R²   : 0.1334
MSE  : 0.8162
RMSE : 0.9034


### Compare all models

In [17]:
# ============================================================
# COMPARE REGRESSION MODELS
# ============================================================

results = [
    {
        "Model": "Linear Regression",
        "R2": linear_r2,
        "MSE": linear_mse,
        "RMSE": linear_rmse
    },
    {
        "Model": "Random Forest",
        "R2": rf_r2,
        "MSE": rf_mse,
        "RMSE": rf_rmse
    }
]


# Add XGBoost if it was successfully trained
if xgboost_available:

    results.append(
        {
            "Model": "XGBoost",
            "R2": xgb_r2,
            "MSE": xgb_mse,
            "RMSE": xgb_rmse
        }
    )


comparison_df = pd.DataFrame(results)


# Sort by R² from highest to lowest
comparison_df = (
    comparison_df
    .sort_values(
        by="R2",
        ascending=False
    )
    .reset_index(drop=True)
)


display(comparison_df)

,Model,R2,MSE,RMSE
0,Random Forest,0.141160,0.808870,0.899372
1,XGBoost,0.133381,0.816196,0.903436
2,Linear Regression,0.096029,0.851374,0.922699


In [18]:
# ============================================================
# SELECT BEST MODEL
# ============================================================

best_model_name = comparison_df.iloc[0]["Model"]

print(
    "Best model:",
    best_model_name
)

Best model: Random Forest


### Select the corresponding model object

In [19]:
# ============================================================
# GET BEST MODEL OBJECT
# ============================================================

if best_model_name == "Linear Regression":

    best_model = linear_model

elif best_model_name == "Random Forest":

    best_model = random_forest

elif best_model_name == "XGBoost":

    best_model = xgboost_model


print(
    "Selected model:",
    type(best_model).__name__
)

Selected model: RandomForestRegressor


#### Save model comparison table

In [20]:
# ============================================================
# SAVE MODEL COMPARISON TABLE
# ============================================================

comparison_path = (
    "../data/processed/regression_model_comparison.csv"
)


comparison_df.to_csv(
    comparison_path,
    index=False
)


print(
    "Model comparison saved successfully."
)

print(
    "File:",
    comparison_path
)

Model comparison saved successfully.
File: ../data/processed/regression_model_comparison.csv


### Create the models directory

In [21]:
# ============================================================
# CREATE MODELS DIRECTORY
# ============================================================

models_dir = Path("../models")

models_dir.mkdir(
    parents=True,
    exist_ok=True
)


print(
    "Models directory ready."
)

Models directory ready.


In [22]:
# ============================================================
# SAVE BEST REGRESSION MODEL
# ============================================================

model_path = (
    models_dir /
    "regression_model.pkl"
)


joblib.dump(
    best_model,
    model_path
)


print(
    "Best regression model saved successfully!"
)

print(
    "Model path:",
    model_path
)

Best regression model saved successfully!
Model path: ../models/regression_model.pkl


### verify the model file

In [23]:
# ============================================================
# VERIFY MODEL FILE
# ============================================================

print(
    "Model file exists:",
    model_path.exists()
)

if model_path.exists():

    model_size_kb = (
        model_path.stat().st_size / 1024
    )

    print(
        f"Model size: {model_size_kb:.2f} KB"
    )

Model file exists: True
Model size: 37079.72 KB


### Load the saved model

In [24]:
# ============================================================
# LOAD SAVED MODEL
# ============================================================

loaded_model = joblib.load(
    model_path
)


print(
    "Saved model loaded successfully."
)

print(
    "Loaded model type:",
    type(loaded_model).__name__
)

Saved model loaded successfully.
Loaded model type: RandomForestRegressor


### Test the same model

In [25]:
# ============================================================
# TEST SAVED MODEL WITH UNSEEN DATA
# ============================================================

sample_X = X_test.iloc[:5]

sample_y = y_test.iloc[:5]


sample_predictions = (
    loaded_model.predict(sample_X)
)


results_sample = pd.DataFrame(
    {
        "Actual_Rating": sample_y.values,
        "Predicted_Rating": sample_predictions
    }
)


results_sample["Prediction_Error"] = (
    results_sample["Actual_Rating"]
    -
    results_sample["Predicted_Rating"]
)


display(results_sample)

,Actual_Rating,Predicted_Rating,Prediction_Error
0,5,4.266474,0.733526
1,5,4.123586,0.876414
2,3,3.472675,-0.472675
3,4,4.434898,-0.434898
4,5,3.974201,1.025799


### Calculate final model metrics

In [26]:
# ============================================================
# FINAL BEST MODEL PERFORMANCE
# ============================================================

best_result = comparison_df.iloc[0]


print("=" * 50)
print("FINAL BEST REGRESSION MODEL")
print("=" * 50)

print(
    "Model:",
    best_result["Model"]
)

print(
    f"R²   : {best_result['R2']:.4f}"
)

print(
    f"MSE  : {best_result['MSE']:.4f}"
)

print(
    f"RMSE : {best_result['RMSE']:.4f}"
)

FINAL BEST REGRESSION MODEL
Model: Random Forest
R²   : 0.1412
MSE  : 0.8089
RMSE : 0.8994


#### Check the regression model features

In [27]:
# -----------------------------------------
# Check regression model features
# -----------------------------------------

regression_model = joblib.load(
    "../models/regression_model.pkl"
)

print(
    "Regression model type:",
    type(regression_model)
)

if hasattr(
    regression_model,
    "feature_names_in_"
):

    print(
        "Number of expected features:",
        len(regression_model.feature_names_in_)
    )

    print(
        regression_model.feature_names_in_
    )

else:

    print(
        "Feature names are not directly available."
    )

Regression model type: <class 'sklearn.ensemble._forest.RandomForestRegressor'>
Number of expected features: 86
['VisitDate' 'VisitYear' 'VisitMonth' 'UserVisitCount'
 'AttractionVisitCount' 'Country_Frequency' 'CityName_Frequency'
 'VisitMode_Business' 'VisitMode_Couples' 'VisitMode_Family'
 'VisitMode_Friends' 'VisitMode_Solo' 'Continent_Africa'
 'Continent_America' 'Continent_Asia' 'Continent_Australia & Oceania'
 'Continent_Europe' 'Region_Australia' 'Region_Caribbean'
 'Region_Central Africa' 'Region_Central America' 'Region_Central Asia'
 'Region_Central Europe' 'Region_East Africa' 'Region_East Asia'
 'Region_Eastern Europe' 'Region_Middle East' 'Region_North Africa'
 'Region_Northern America' 'Region_Northern Europe' 'Region_Oceania'
 'Region_South America' 'Region_South Asia' 'Region_South East Asia'
 'Region_Southern Africa' 'Region_Southern Europe' 'Region_Unknown'
 'Region_West Africa' 'Region_Western Europe'
 'AttractionType_Ancient Ruins' 'AttractionType_Ballets'
 'Attrac